# ทดลอง Wayu TTS: “ฉันไม่ได้คุยโม้” 1 คลิป

MMS อ่านคำว่า “โม้” ไม่ตรงตามที่ฟังตรวจ แม้ลองเปลี่ยน seed แล้ว จึงทดลอง **ตัวสร้างเสียงอีกโมเดล** ด้วยข้อความ Train เดิม ไม่ใช่การ fine-tune AASIST/RawNet2 และไม่ใช่ voice cloning

เลือก **Select Kernel → Python Environments → .venv-wayu (Python 3.11)** แล้วรันจากบนลงล่าง ชุดนี้แยกจาก `.venv` เดิม เพราะ tltk ต้องใช้ scikit-learn 1.2.x ซึ่งไม่มี wheel สำหรับ Python 3.12 ดูวิธีเตรียมใน [README](README.md#wayu-pilot)

ใช้ [Wayu-Paxa-TTS-Edge](https://huggingface.co/wayu-ai/wayu-paxa-tts-edge) และ [โค้ดผู้พัฒนา](https://github.com/wayu-research/wayu-tts-inference) ที่ล็อก revision ไว้ น้ำหนัก CC-BY-NC-4.0 / โค้ด Apache-2.0 ต้องปฏิบัติตาม [เงื่อนไขการใช้](https://www.wayuresearch.org/terms) รวมถึงระบุว่าเป็นเสียงสังเคราะห์ ไม่สวมรอยผู้พูดจริง

มีเสียงแบบกำหนดไว้ให้เลือก แต่ครั้งนี้เลือกแค่ `m_young_clear` ไม่ป้อนเสียงคนให้ TTS ชื่อ voice **ไม่ใช่ข้อมูลอายุ/เพศที่ยืนยันของบุคคล** ผลลัพธ์ native 24 kHz เก็บแยกจาก MMS ไม่ resample/crop/เติม noise และไม่แก้ manifest เดิม

In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import importlib.metadata as metadata
import pandas as pd
from IPython.display import Audio, display, Markdown

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("เปิด notebook ภายใน project ก่อน")
    PROJECT_ROOT = PROJECT_ROOT.parent

expected_python = PROJECT_ROOT / ".venv-wayu/Scripts/python.exe"
if Path(sys.executable).resolve() != expected_python.resolve():
    raise RuntimeError("เลือก Kernel .venv-wayu (Python 3.11) ไม่ใช่ .venv เดิม")
assert sys.version_info[:2] == (3, 11)

PILOT_DIR = PROJECT_ROOT / "data/processed/cvtts/pilot_v1"
TRAIN_MANIFEST = PILOT_DIR / "manifests/real_train.csv"
split_report = json.loads((PILOT_DIR / "qc/pilot_split_report.json").read_text(encoding="utf-8"))
train_sha = hashlib.sha256(TRAIN_MANIFEST.read_bytes()).hexdigest()
assert train_sha == split_report["manifest_sha256"]["real_train.csv"]

df_train = pd.read_csv(TRAIN_MANIFEST, dtype="string", keep_default_na=False)
assert len(df_train) == 80 and df_train["split"].eq("train").all()
chosen = df_train[df_train["sample_id"].eq("common_voice_th_25669007")]
assert len(chosen) == 1, "ต้องมีข้อความต้นทางรายการเดียวใน Train"
source_row = chosen.iloc[0]
assert source_row["label"] == "bonafide"
tts_text = source_row["sentence"]
assert tts_text == "ฉันไม่ได้คุยโม้"
real_path = (PROJECT_ROOT / source_row["audio_path"]).resolve()
assert real_path.is_relative_to((PROJECT_ROOT / "data/raw/common_voice").resolve())
assert hashlib.sha256(real_path.read_bytes()).hexdigest() == source_row["audio_file_sha256"]
display(chosen[["sample_id", "sentence", "split"]])
print("Python:", sys.version.split()[0])
print("ไม่ได้อ่านข้อความจาก Dev/Test")

## 1. โหลดโมเดลสำเร็จรูปและตรวจคำอ่าน

โหลดจากโฟลเดอร์ในเครื่องเมื่อไฟล์ครบ ครั้งแรกดาวน์โหลดเฉพาะ config, weights, voice เดียว และ model card ประมาณ 328 MB ไม่ใช้ reference audio

โมเดลแปลงข้อความเป็น phonemes ก่อนสร้างเสียง เราบันทึก phonemes และ trace ไว้ตรวจว่ามีส่วนที่ frontend รายงานว่าหายหรือไม่ แต่ trace ที่ผ่าน **ยังไม่ยืนยันว่าคำอ่านถูกทุกคำ** ต้องฟังจริง ไม่เปลี่ยนข้อความหรือแก้ phonemes เพื่อบังคับให้ผ่านใน trial นี้

In [ ]:
import torch
from huggingface_hub import snapshot_download
from wayu_tts import ThaiTTS

MODEL_ID = "wayu-ai/wayu-paxa-tts-edge"
MODEL_REVISION = "8196688df56a8d08ccfa186505343adff412ab6b"
CODE_REVISION = "07ab898d516649313926f0b843b809a962f993ad"
VOICE = "m_young_clear"
SEED = 42
SPEED = 1.0
MODEL_DIR = PROJECT_ROOT / "checkpoints/tts/wayu-paxa-tts-edge" / MODEL_REVISION
MODEL_FILES = ["config.json", "model.pth", f"voices/{VOICE}.pt", "README.md"]

# ตรวจว่าแพ็กเกจมาจาก source revision ที่กำหนด ไม่ติดตั้ง source อัตโนมัติใน notebook
dist = metadata.distribution("wayu-tts")
direct_url = json.loads(dist.read_text("direct_url.json") or "{}")
assert CODE_REVISION in direct_url.get("url", ""), "ติดตั้งแพ็กเกจตาม requirements ของ trial นี้ก่อน"
if not all((MODEL_DIR / name).is_file() for name in MODEL_FILES):
    snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION,
                      local_dir=MODEL_DIR, allow_patterns=MODEL_FILES)

torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
tts = ThaiTTS.from_pretrained(MODEL_DIR, device="cpu")
trace = tts.g2p.trace(tts_text)
phoneme_chunks = tts.phonemize(tts_text)
assert trace.is_clean and phoneme_chunks, "frontend รายงานส่วนที่หาย ต้องตรวจ ไม่สร้างต่อ"
assert all(tts.g2p.count_tokens(chunk) > 0 for chunk in phoneme_chunks)

print("โมเดล:", MODEL_ID)
print("voice:", VOICE)
print("native sample rate:", tts.sample_rate)
print("parameters:", sum(p.numel() for p in tts.model.parameters()))
print("phonemes:", phoneme_chunks)
print("dropped words:", trace.dropped_words)
print("dropped symbols:", sorted(trace.dropped_symbols))

## 2. สร้างหนึ่งคลิป และตรวจการรันซ้ำ

ใช้ CPU, seed 42, speed 1.0 และค่าปกติของ voice นี้ ความยาวตามข้อความ ไม่บังคับให้เป็น 4.04 วินาที เราสร้างซ้ำใน memory เพื่อเช็กว่า waveform ตรงกัน แต่บันทึกเพียงหนึ่งไฟล์

ตรวจ waveform ไม่ว่าง ไม่มี NaN/Inf และไม่เงียบเกือบศูนย์ การตรวจนี้ **ไม่แทนการฟังความถูกต้องหรือคุณภาพ**

In [ ]:
import numpy as np
import time

started = time.perf_counter()
waveform = tts(tts_text, voice=VOICE, speed=SPEED, seed=SEED)
generation_seconds = time.perf_counter() - started
waveform = np.asarray(waveform, dtype=np.float32)
assert waveform.ndim == 1 and waveform.size > 0
assert np.isfinite(waveform).all()
assert float(np.abs(waveform).max()) > 1e-8
repeat = np.asarray(tts(tts_text, voice=VOICE, speed=SPEED, seed=SEED), dtype=np.float32)
assert np.array_equal(waveform, repeat), "รันซ้ำไม่ตรงกัน ต้องตรวจ environment ก่อนบันทึก"

sample_rate = int(tts.sample_rate)
assert sample_rate == 24000
print("ความยาว:", round(waveform.size / sample_rate, 3), "วินาที")
print("sample rate:", sample_rate, "Hz")
print("CPU generation:", round(generation_seconds, 3), "วินาที")
print("peak:", float(np.abs(waveform).max()))
print("รันซ้ำแล้ว waveform ตรงกัน")

## 3. บันทึกเสียงใหม่และที่มา แยกจาก MMS

บันทึก WAV float32 โดยไม่ normalize/crop/เพิ่ม noise มีรายงาน revision, seed, voice, frontend, config, model hashes, software และ lineage ของข้อความ **ไม่ถือว่าผู้พูด TTS คือเจ้าของคลิป Common Voice**

ถ้าไฟล์ชื่อเดิมมีข้อมูลต่างกันจะหยุด ไม่เขียนทับ ทั้งเสียง/โมเดล/รายงานในเครื่องไม่เข้า Git สถานะการฟังยังเป็น `pending` อย่าเพิ่มเข้าชุดฝึกเพียงเพราะรันสำเร็จ

In [ ]:
import io
import platform
from scipy.io import wavfile
import soundfile as sf

fake_id = f"wayu_paxa_{source_row['sample_id']}_{VOICE}_seed{SEED}"
FAKE_PATH = PILOT_DIR / "native_tts/wayu/train" / f"{fake_id}.wav"
REPORT_PATH = PILOT_DIR / "qc/tts_samples" / f"{fake_id}.json"
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, waveform)
wav_bytes = buffer.getvalue()
decoded, decoded_sr = sf.read(io.BytesIO(wav_bytes), dtype="float32")
assert decoded_sr == sample_rate and np.array_equal(decoded, waveform)

report = {
    "sample_id": fake_id,
    "audio_path": FAKE_PATH.relative_to(PROJECT_ROOT).as_posix(),
    "label": "spoof", "split": "train", "language": "th",
    "experiment": "clip3_wayu_fixed_voice_v1",
    "source_sample_id": source_row["sample_id"],
    "source_text_key": source_row["text_key"],
    "source_train_manifest_sha256": train_sha,
    "source_audio_file_sha256": source_row["audio_file_sha256"],
    "tts_text": tts_text,
    "reference_audio_used": False,
    "generator_model": MODEL_ID, "generator_revision": MODEL_REVISION,
    "generator_code_repository": "https://github.com/wayu-research/wayu-tts-inference",
    "generator_code_revision": CODE_REVISION,
    "generator_license": "CC-BY-NC-4.0", "generator_code_license": "Apache-2.0",
    "voice": VOICE, "generation_seed": SEED,
    "device": "cpu", "cpu_threads": 2, "deterministic_algorithms": True,
    "settings": {"speed": SPEED, "calibrated_voice_speed": tts.config.voice_speed(VOICE)},
    "phoneme_chunks": phoneme_chunks,
    "frontend_trace": {"dropped_words": trace.dropped_words,
                       "dropped_symbols": sorted(trace.dropped_symbols),
                       "is_clean": trace.is_clean,
                       "not_a_human_pronunciation_verdict": True},
    "model_config": json.loads((MODEL_DIR / "config.json").read_text(encoding="utf-8")),
    "sample_rate": sample_rate, "channels": 1,
    "samples": int(waveform.size), "duration_s": waveform.size / sample_rate,
    "wav_subtype": "FLOAT", "peak": float(np.abs(waveform).max()),
    "audio_file_sha256": hashlib.sha256(wav_bytes).hexdigest(),
    "waveform_sha256": hashlib.sha256(waveform.astype("<f4", copy=False).tobytes()).hexdigest(),
    "repeat_generation_waveform_equal": True,
    "model_file_sha256": {
        name: hashlib.sha256((MODEL_DIR / name).read_bytes()).hexdigest()
        for name in MODEL_FILES
    },
    "python": platform.python_version(),
    "software": {p: metadata.version(p) for p in [
        "wayu-tts", "kokoro", "tltk", "pythainlp", "misaki", "torch",
        "transformers", "huggingface-hub", "numpy", "scipy", "scikit-learn",
        "gensim", "soundfile"
    ]},
    "comparison_mms_sample_id": "mms_tha_common_voice_th_25669007_seed42",
    "manual_review": {"status": "pending", "text_matches": None, "notes": ""},
}
prepared = {FAKE_PATH: wav_bytes,
            REPORT_PATH: (json.dumps(report, ensure_ascii=False, indent=2) + "\n").encode("utf-8")}
# ตรวจทุกไฟล์ก่อนเริ่มเขียน; ไม่แก้ไฟล์เดิม/manifest/human reviews
for path, content in prepared.items():
    if path.exists() and path.read_bytes() != content:
        raise FileExistsError(f"ไม่เขียนทับผลเดิม: {path}")
for path, content in prepared.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("xb") as stream:
            stream.write(content)
    print(path.relative_to(PROJECT_ROOT).as_posix())

## 4. ฟังเทียบ: คน → MMS → Wayu

ทุกตัวอ่านข้อความ **“ฉันไม่ได้คุยโม้”** แต่เสียง TTS ไม่ใช่การเลียนแบบเสียงคนต้นทาง

กดเล่นทีละตัว โดยเฉพาะคำสุดท้ายว่าได้ยินเป็น **โม้** หรือยังเป็น **โหมด/โมด/โหวด** และมีคำหาย/เสียงขาดหรือไม่ ค่า `normalize=False` ไม่ปรับระดับเสียงเพื่อกลบความต่าง ถ้าได้ยินถูกจึงค่อยพิจารณาทดลองหลายข้อความ ไม่ได้หมายความว่าโมเดลทั้งชุดผ่านจากคลิปเดียว

In [ ]:
display(Markdown("**1. เสียงคนต้นทาง**"))
display(Audio(filename=str(real_path), normalize=False))

mms_report_path = PILOT_DIR / "qc/tts_samples/mms_tha_common_voice_th_25669007_seed42.json"
if mms_report_path.is_file():
    mms_report = json.loads(mms_report_path.read_text(encoding="utf-8"))
    mms_path = (PROJECT_ROOT / mms_report["audio_path"]).resolve()
    assert mms_path.is_relative_to(PILOT_DIR.resolve())
    assert hashlib.sha256(mms_path.read_bytes()).hexdigest() == mms_report["audio_file_sha256"]
    display(Markdown("**2. MMS เดิม seed 42 — ผู้ใช้ฟังว่า โม้ → โหมด**"))
    display(Audio(filename=str(mms_path), normalize=False))
else:
    print("ไม่มีตัวอย่าง MMS เดิมในเครื่องนี้ ไม่สร้างใหม่อัตโนมัติ")

assert hashlib.sha256(FAKE_PATH.read_bytes()).hexdigest() == report["audio_file_sha256"]
display(Markdown("**3. Wayu ใหม่ m_young_clear seed 42 — ยังรอฟังตรวจ**"))
display(Audio(filename=str(FAKE_PATH), normalize=False))
print("ช่วยบอก: คำว่า โม้ ถูกไหม / คำครบไหม / มีเสียงวนหรือขาดไหม")